
# aio2.ipynb

통합 AIO 판정 노트북입니다.

목표:

- **스토킹**: 사람 track 경로, 같은 방향 추적, 거리 유지, 장시간 동행 패턴에 비중
- **싸움**: YOLO11-pose + optical flow + 근접 상호작용 + 슬라이딩 윈도우 peak score
- **주행/인도침범**: YOLO-World 객체 보강 + SegFormer 도로/인도 segmentation + 침범 기준
- 최종 결과는 하나로 덮어쓰지 않고 `action_label`, `stalking_label`, `fight_label`, `riding_label`, `final_tags`를 모두 저장

입력 영상 루트는 사용자가 준 경로를 그대로 사용합니다.


추가 개선: 이륜차/킥보드 검출 강화, YOLO/YOLO-World confidence 완화, peak 기반 주행 점수, 디버그 출력 포함.

In [1]:

# ============================================================
# Cell 1. 경로 및 전체 설정
# ============================================================
from pathlib import Path
import os, json, math, time, gc, warnings
from collections import Counter, defaultdict

VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

AIO_DIR = OUTPUT_ROOT / "aio2"
AIO_FEATURE_DIR = AIO_DIR / "aio_features"
AIO_FINAL_DIR = AIO_DIR / "final_aio"
AIO_DEBUG_DIR = AIO_DIR / "debug"

for d in [AIO_DIR, AIO_FEATURE_DIR, AIO_FINAL_DIR, AIO_DEBUG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}

# 전체 처리 시 None. 테스트 시 2~5 정도로 제한 가능.
MAX_VIDEOS = None
MAX_FRAMES_PER_VIDEO = None

# 샘플링 설정
# aio2: 이륜차/킥보드가 짧게 등장해도 잡히도록 더 촘촘하게 샘플링
FRAME_INTERVAL = 5
WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 1.5

# 모델 설정
BASE_YOLO_MODEL_PATH = "yolov8n.pt"
POSE_MODEL_CANDIDATES = ["yolo11n-pose.pt", "yolov8n-pose.pt"]
YOLO_WORLD_MODEL_CANDIDATES = ["yolov8s-world.pt", "yolov8m-world.pt"]

# aio2: 작은 오토바이/킥보드 검출 강화용 추론 이미지 크기
YOLO_IMGSZ = 960
WORLD_IMGSZ = 960

# SegFormer 설정: CUDA면 b5, CPU면 b0 fallback
SEG_MODEL_ID_GPU = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"

# Cityscapes label 기준
ROAD_ID = 0
SIDEWALK_ID = 1

# threshold: 결과가 너무 포괄적이면 올리고, 너무 놓치면 내리면 됨
FIGHT_THRESHOLD = 0.44
STALKING_THRESHOLD = 0.42
RIDE_DETECTION_THRESHOLD = 0.035
RIDE_SIDEWALK_THRESHOLD = 0.24
RIDE_SIDEWALK_CANDIDATE_THRESHOLD = 0.12

# 검출 confidence
PERSON_MIN_CONF = 0.12
VEHICLE_MIN_CONF = 0.06
WORLD_MIN_CONF = 0.015
POSE_MIN_CONF = 0.15

# 트래킹/행동 기준
TRACK_MAX_DIST = 150
MIN_TRACK_LEN_FOR_STALKING = 5
FOLLOW_DISTANCE_MIN = 40
FOLLOW_DISTANCE_MAX = 450
FOLLOW_DIRECTION_COS_THRESHOLD = 0.55
FOLLOW_DISTANCE_STABILITY = 130

# optical flow 기준
FLOW_RESIZE_WIDTH = 480
FLOW_MAG_STRONG = 3.0

# 주행/침범 기준
TWO_WHEELER_CLASSES = {"bicycle", "motorcycle"}
WORLD_TWO_WHEELER_HINTS = {
    "motorcycle", "motorbike", "moped", "scooter", "electric scooter",
    "kick scooter", "kickboard", "electric kickboard", "bicycle", "bike",
    "delivery motorcycle", "e-scooter"
}
RIDER_CLASSES = {"person"}
SIDEWALK_PATCH_RADIUS = 18
BOTTOM_POINT_Y_RATIO = 0.92
BOTTOM_BAND_Y1 = 0.70
BOTTOM_BAND_Y2 = 1.00
BOTTOM_BAND_X_MARGIN = 0.10

# aio2: rider-like 거리/겹침 조건 완화
RIDER_PAIR_MAX_DIST = 360
RIDER_PAIR_MIN_IOU = 0.005

# 헬멧 모델은 선택. 별도 학습 모델 있으면 경로 입력.
HELMET_MODEL_PATH = None
HELMET_CONF = 0.15

print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("AIO_DIR:", AIO_DIR)

if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"원본 영상 루트를 찾지 못했습니다: {VIDEO_ROOT}")


VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
AIO_DIR: C:\Users\우성한\outputs\prediction\aio2


In [2]:

# ============================================================
# Cell 2. 의존성 확인 및 모델 로드
# ============================================================
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    from tqdm import tqdm
    print("tqdm OK")
except Exception as e:
    missing.append("tqdm")
    print("tqdm import failed:", repr(e))

try:
    import torch
    print("torch OK, cuda:", torch.cuda.is_available())
except Exception as e:
    torch = None
    missing.append("torch")
    print("torch import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics YOLO OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

if missing:
    print("\n설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy tqdm ultralytics torch transformers")

if YOLO is None:
    raise RuntimeError("ultralytics YOLO를 사용할 수 없습니다.")

# 기본 YOLO
base_yolo = YOLO(BASE_YOLO_MODEL_PATH)
print("base_yolo loaded:", BASE_YOLO_MODEL_PATH)

# pose 모델: yolo11n-pose 우선, 실패하면 yolov8n-pose
pose_model = None
pose_model_name = None
for cand in POSE_MODEL_CANDIDATES:
    try:
        pose_model = YOLO(cand)
        pose_model_name = cand
        print("pose_model loaded:", cand)
        break
    except Exception as e:
        print("pose_model load failed:", cand, repr(e))

# YOLO-World: 선택. 실패해도 계속 진행.
world_model = None
world_model_name = None
world_classes = [
    "person",
    "rider",
    "motorcycle",
    "motorbike",
    "moped",
    "scooter",
    "electric scooter",
    "kick scooter",
    "kickboard",
    "electric kickboard",
    "bicycle",
    "bike",
    "delivery motorcycle",
    "helmet",
    "no helmet",
    "road",
    "sidewalk",
    "crosswalk",
    "curb",
]
for cand in YOLO_WORLD_MODEL_CANDIDATES:
    try:
        world_model = YOLO(cand)
        world_model_name = cand
        try:
            world_model.set_classes(world_classes)
        except Exception as e:
            print("world_model set_classes failed, continue:", repr(e))
        print("world_model loaded:", cand)
        break
    except Exception as e:
        print("world_model load failed:", cand, repr(e))

# SegFormer: 선택. 실패해도 주행은 bbox 기반 후보로 계속 진행.
seg_processor = None
seg_model = None
seg_device = None
seg_model_id = None
try:
    from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
    if torch is not None:
        seg_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        seg_model_id = SEG_MODEL_ID_GPU if seg_device.type == "cuda" else SEG_MODEL_ID_CPU
        seg_processor = SegformerImageProcessor.from_pretrained(seg_model_id)
        seg_model = SegformerForSemanticSegmentation.from_pretrained(seg_model_id).to(seg_device)
        seg_model.eval()
        print("SegFormer loaded:", seg_model_id, "device:", seg_device)
except Exception as e:
    print("SegFormer load failed. sidewalk segmentation fallback will be limited:", repr(e))
    seg_processor = None
    seg_model = None

# 헬멧 모델 선택
helmet_model = None
if HELMET_MODEL_PATH:
    try:
        helmet_model = YOLO(HELMET_MODEL_PATH)
        print("helmet_model loaded:", HELMET_MODEL_PATH)
    except Exception as e:
        print("helmet_model load failed:", repr(e))

MODEL_STATUS = {
    "base_yolo": BASE_YOLO_MODEL_PATH,
    "pose_model": pose_model_name,
    "world_model": world_model_name,
    "segformer": seg_model_id,
    "helmet_model": HELMET_MODEL_PATH if helmet_model is not None else None,
}
print("MODEL_STATUS:", MODEL_STATUS)


cv2 OK
numpy OK
tqdm OK
torch OK, cuda: True
ultralytics YOLO OK
base_yolo loaded: yolov8n.pt
pose_model loaded: yolo11n-pose.pt
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command '"C:\Users\\cv_env\Scripts\python.exe" -m pip install --no-cache-dir "git+https://github.com/ultralytics/CLIP.git" ' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command '"C:\Users\\cv_env\Scripts\python.exe" -m pip install --no-cache-dir "git+https://github.com/ultralytics/CLIP.git" ' returned non-zero exit status 1.
WARNING requirements:  Command '"C:\Users\\cv_env\Scripts\python.exe" -m pip install --no-cache-dir "git+https://github.com/ultralytics/CLIP.git" ' returned non-zero exit status 1.
  Cloning https://github.com/ultralytics/CLIP.git to .\AppData\Local\Temp\pip-req-build-4bvysutc
  ERROR: Error [WinError 2]      while executing command git version
ERROR: Cannot find command 'git' -

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

SegFormer loaded: nvidia/segformer-b5-finetuned-cityscapes-1024-1024 device: cuda
MODEL_STATUS: {'base_yolo': 'yolov8n.pt', 'pose_model': 'yolo11n-pose.pt', 'world_model': 'yolov8s-world.pt', 'segformer': 'nvidia/segformer-b5-finetuned-cityscapes-1024-1024', 'helmet_model': None}


In [3]:

# ============================================================
# Cell 3. 영상 탐색
# ============================================================
video_paths = sorted([p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix in VIDEO_EXTS])
if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:30]:
    print(p)

if not video_paths:
    raise RuntimeError("원천데이터 안에서 영상 파일을 찾지 못했습니다.")


찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [4]:

# ============================================================
# Cell 4. 공통 유틸 함수
# ============================================================
def safe_stem_from_path(path: Path, root: Path):
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts)


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def bbox_center(b):
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / union if union > 0 else 0


def dist(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def vec(a, b):
    return (b[0] - a[0], b[1] - a[1])


def norm(v):
    return math.sqrt(v[0] ** 2 + v[1] ** 2)


def cosine(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 <= 1e-6 or n2 <= 1e-6:
        return 0
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)


def video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"fps": None, "width": None, "height": None, "frame_count": None}
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        "fps": float(fps) if fps and fps > 0 else None,
        "width": width,
        "height": height,
        "frame_count": frame_count,
    }


def yolo_objects(result, names, min_conf=0.0):
    objs = []
    if result is None or result.boxes is None:
        return objs
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = names[cls_id] if isinstance(names, dict) else str(cls_id)
        conf = float(box.conf[0])
        if conf < min_conf:
            continue
        bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
        objs.append({
            "class": cls_name,
            "confidence": round(conf, 4),
            "bbox": bbox,
            "center": [round(v, 2) for v in bbox_center(bbox)],
            "area": round(bbox_area(bbox), 2),
        })
    return objs


def extract_pose_people(result):
    people = []
    if result is None or result.boxes is None:
        return people
    boxes = result.boxes
    kpts = getattr(result, "keypoints", None)
    xy_data = None
    conf_data = None
    if kpts is not None:
        try:
            xy_data = kpts.xy.cpu().numpy()
            conf_data = kpts.conf.cpu().numpy() if kpts.conf is not None else None
        except Exception:
            xy_data = None
            conf_data = None
    for i, box in enumerate(boxes):
        try:
            conf = float(box.conf[0])
            bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
        except Exception:
            continue
        item = {"bbox": bbox, "confidence": round(conf, 4), "center": [round(v,2) for v in bbox_center(bbox)]}
        if xy_data is not None and i < len(xy_data):
            pts = xy_data[i].tolist()
            if conf_data is not None and i < len(conf_data):
                pc = conf_data[i].tolist()
            else:
                pc = [None] * len(pts)
            item["keypoints"] = [
                {"x": round(float(p[0]), 2), "y": round(float(p[1]), 2), "conf": None if c is None else round(float(c), 4)}
                for p, c in zip(pts, pc)
            ]
        people.append(item)
    return people


def compute_pose_agitation(pose_people):
    """팔/상체 keypoint 분산을 이용한 간단한 자세 불안정도. keypoint가 없으면 0."""
    vals = []
    # COCO keypoints: shoulders 5,6 elbows 7,8 wrists 9,10 hips 11,12
    for p in pose_people:
        k = p.get("keypoints") or []
        if len(k) < 13:
            continue
        important = [5,6,7,8,9,10,11,12]
        pts = [(k[i]["x"], k[i]["y"], k[i].get("conf")) for i in important if k[i].get("conf") is None or k[i].get("conf",0) >= POSE_MIN_CONF]
        if len(pts) < 4:
            continue
        xs = [a[0] for a in pts]
        ys = [a[1] for a in pts]
        spread = (max(xs)-min(xs)) + (max(ys)-min(ys))
        h = max(1, p["bbox"][3] - p["bbox"][1])
        vals.append(spread / h)
    return float(np.mean(vals)) if vals else 0.0


def resize_for_flow(gray):
    h, w = gray.shape[:2]
    if w <= FLOW_RESIZE_WIDTH:
        return gray, 1.0
    scale = FLOW_RESIZE_WIDTH / w
    new_h = max(1, int(h * scale))
    return cv2.resize(gray, (FLOW_RESIZE_WIDTH, new_h)), scale


def optical_flow_stats(prev_gray, gray):
    if prev_gray is None or gray is None:
        return {"mean_mag": 0.0, "p90_mag": 0.0, "strong_ratio": 0.0}
    g1, s1 = resize_for_flow(prev_gray)
    g2, s2 = resize_for_flow(gray)
    if g1.shape != g2.shape:
        g2 = cv2.resize(g2, (g1.shape[1], g1.shape[0]))
    flow = cv2.calcOpticalFlowFarneback(g1, g2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    return {
        "mean_mag": round(float(np.mean(mag)), 4),
        "p90_mag": round(float(np.percentile(mag, 90)), 4),
        "strong_ratio": round(float(np.mean(mag >= FLOW_MAG_STRONG)), 4),
    }


def get_seg_mask(frame_bgr):
    if seg_model is None or seg_processor is None or torch is None:
        return None
    try:
        image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        inputs = seg_processor(images=image_rgb, return_tensors="pt")
        inputs = {k: v.to(seg_device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = seg_model(**inputs)
        logits = outputs.logits
        upsampled = torch.nn.functional.interpolate(
            logits,
            size=image_rgb.shape[:2],
            mode="bilinear",
            align_corners=False,
        )
        pred = upsampled.argmax(dim=1)[0].detach().cpu().numpy().astype("uint8")
        return pred
    except Exception as e:
        return None


def patch_label_ratio(mask, x, y, radius=18):
    if mask is None:
        return {"road_ratio": None, "sidewalk_ratio": None}
    h, w = mask.shape[:2]
    x = int(clamp(x, 0, w-1)); y = int(clamp(y, 0, h-1))
    x1, x2 = clamp(x-radius,0,w-1), clamp(x+radius,0,w-1)
    y1, y2 = clamp(y-radius,0,h-1), clamp(y+radius,0,h-1)
    patch = mask[int(y1):int(y2)+1, int(x1):int(x2)+1]
    if patch.size == 0:
        return {"road_ratio": 0.0, "sidewalk_ratio": 0.0}
    return {
        "road_ratio": round(float(np.mean(patch == ROAD_ID)), 4),
        "sidewalk_ratio": round(float(np.mean(patch == SIDEWALK_ID)), 4),
    }


def bbox_bottom_point(bbox):
    x1, y1, x2, y2 = bbox
    x = (x1 + x2) / 2
    y = y1 + (y2 - y1) * BOTTOM_POINT_Y_RATIO
    return x, y


In [5]:

# ============================================================
# Cell 5. AIO 프레임 특징 JSON 생성
# ============================================================
def create_aio_features_for_video(video_path, output_json_path):
    meta = video_meta(video_path)
    fps = meta.get("fps") or 30.0
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    frames = []
    frame_idx = 0
    sampled_idx = 0
    prev_gray = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if MAX_FRAMES_PER_VIDEO is not None and frame_idx >= MAX_FRAMES_PER_VIDEO:
            break

        if frame_idx % FRAME_INTERVAL == 0:
            t = frame_idx / fps if fps else sampled_idx
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            flow = optical_flow_stats(prev_gray, gray)
            prev_gray = gray

            # 기본 YOLO 객체 탐지
            base_result = base_yolo(frame, verbose=False, imgsz=YOLO_IMGSZ, conf=0.01)[0]
            base_objects = yolo_objects(base_result, base_yolo.names, min_conf=0.0)

            persons = [o for o in base_objects if o["class"] == "person" and o["confidence"] >= PERSON_MIN_CONF]
            base_tw = [o for o in base_objects if o["class"] in TWO_WHEELER_CLASSES and o["confidence"] >= VEHICLE_MIN_CONF]

            # YOLO-World 객체 보강
            world_objects = []
            if world_model is not None:
                try:
                    wr = world_model(frame, verbose=False, imgsz=WORLD_IMGSZ, conf=WORLD_MIN_CONF)[0]
                    world_objects = yolo_objects(wr, world_model.names, min_conf=WORLD_MIN_CONF)
                except Exception as e:
                    world_objects = []

            world_tw = []
            for o in world_objects:
                cls = str(o.get("class", "")).lower()
                if cls in WORLD_TWO_WHEELER_HINTS or any(h in cls for h in ["scooter", "bike", "motor"]):
                    world_tw.append(o)

            # pose
            pose_people = []
            pose_agitation = 0.0
            if pose_model is not None:
                try:
                    pr = pose_model(frame, verbose=False)[0]
                    pose_people = extract_pose_people(pr)
                    pose_agitation = compute_pose_agitation(pose_people)
                except Exception:
                    pose_people = []
                    pose_agitation = 0.0

            # segmentation
            seg_mask = get_seg_mask(frame)

            two_wheelers = base_tw + world_tw
            sidewalk_evidence = []
            for obj in two_wheelers:
                bx = obj.get("bbox")
                if not bx:
                    continue
                x, y = bbox_bottom_point(bx)
                ratios = patch_label_ratio(seg_mask, x, y, SIDEWALK_PATCH_RADIUS)
                sidewalk_evidence.append({
                    "class": obj.get("class"),
                    "confidence": obj.get("confidence"),
                    "bbox": bx,
                    "bottom_point": [round(x,2), round(y,2)],
                    **ratios,
                })

            # rider_like: 사람과 이륜차가 가까운지/겹치는지
            rider_pairs = []
            for p in persons:
                for tw in two_wheelers:
                    d = dist(p["center"], tw["center"])
                    iou = bbox_iou(p["bbox"], tw["bbox"])
                    if d < RIDER_PAIR_MAX_DIST or iou > RIDER_PAIR_MIN_IOU:
                        rider_pairs.append({
                            "person_bbox": p["bbox"],
                            "two_wheeler_bbox": tw["bbox"],
                            "two_wheeler_class": tw["class"],
                            "distance": round(d, 2),
                            "iou": round(iou, 4),
                        })

            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "time_sec": round(t, 3),
                "base_objects": base_objects,
                "world_objects": world_objects,
                "persons": persons,
                "two_wheelers": two_wheelers,
                "rider_pairs": rider_pairs,
                "pose_people": pose_people,
                "pose_agitation": round(float(pose_agitation), 4),
                "flow": flow,
                "sidewalk_evidence": sidewalk_evidence,
                "segformer_used": seg_mask is not None,
            })
            sampled_idx += 1

        frame_idx += 1

    cap.release()

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "meta": meta,
        "settings": {
            "frame_interval": FRAME_INTERVAL,
            "window_seconds": WINDOW_SECONDS,
            "window_stride_seconds": WINDOW_STRIDE_SECONDS,
            "model_status": MODEL_STATUS,
        },
        "frames": frames,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data

created = 0
used = 0
failed = []

for video_path in tqdm(video_paths, desc="AIO feature JSON"):
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    out_path = AIO_FEATURE_DIR / f"{stem}.json"
    if out_path.exists():
        used += 1
        continue
    try:
        create_aio_features_for_video(video_path, out_path)
        created += 1
    except Exception as e:
        failed.append({"video": str(video_path), "error": repr(e)})

print("="*80)
print("AIO feature 생성 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used)
print("실패:", len(failed))
if failed:
    for item in failed[:20]:
        print(item)


AIO feature JSON: 100%|████████████████████████████████████████████████████████████████| 22/22 [31:40<00:00, 86.39s/it]

AIO feature 생성 완료
전체 영상: 22
생성: 22
기존 사용: 0
실패: 0


In [6]:

# ============================================================
# Cell 6. 트랙 생성, 스토킹 점수, 싸움 점수, 주행 점수 계산 함수
# ============================================================
def assign_tracks(frames):
    next_id = 1
    active = {}  # tid -> last center
    track_history = defaultdict(list)

    for fr in frames:
        persons = fr.get("persons", [])
        assigned = set()
        current_active = {}

        for p in persons:
            c = tuple(p.get("center", bbox_center(p["bbox"])))
            best_tid, best_d = None, None
            for tid, last_c in active.items():
                if tid in assigned:
                    continue
                d = dist(c, last_c)
                if best_d is None or d < best_d:
                    best_tid, best_d = tid, d
            if best_tid is not None and best_d <= TRACK_MAX_DIST:
                tid = best_tid
            else:
                tid = next_id
                next_id += 1
            assigned.add(tid)
            current_active[tid] = c
            item = dict(p)
            item["track_id"] = tid
            item["time_sec"] = fr.get("time_sec")
            item["frame_id"] = fr.get("frame_id")
            track_history[tid].append(item)

        active = current_active

    return dict(track_history)


def track_motion(track):
    pts = [tuple(x["center"]) for x in track]
    if len(pts) < 2:
        return {"length": len(pts), "distance": 0, "displacement": 0, "direction_changes": 0, "mean_speed": 0}
    segs = [dist(pts[i-1], pts[i]) for i in range(1, len(pts))]
    displacement = dist(pts[0], pts[-1])
    direction_changes = 0
    for i in range(2, len(pts)):
        v1 = vec(pts[i-2], pts[i-1])
        v2 = vec(pts[i-1], pts[i])
        if cosine(v1, v2) < 0.2 and norm(v1) > 10 and norm(v2) > 10:
            direction_changes += 1
    total_time = max(1e-6, track[-1]["time_sec"] - track[0]["time_sec"])
    return {
        "length": len(pts),
        "distance": round(sum(segs), 3),
        "displacement": round(displacement, 3),
        "direction_changes": direction_changes,
        "mean_speed": round(sum(segs) / total_time, 3),
    }


def compute_stalking_score(frames, tracks):
    # 같은 시간대에 두 track이 일정 거리로 같은 방향 이동하면 점수 증가
    track_ids = [tid for tid, tr in tracks.items() if len(tr) >= MIN_TRACK_LEN_FOR_STALKING]
    follow_events = 0
    total_pair_events = 0
    stable_events = 0
    long_track_bonus = 0
    best_pair = None
    best_pair_score = 0

    for tid in track_ids:
        m = track_motion(tracks[tid])
        if m["length"] >= MIN_TRACK_LEN_FOR_STALKING and m["displacement"] > 80:
            long_track_bonus += 1

    # frame time -> track observations
    by_time = defaultdict(dict)
    for tid, tr in tracks.items():
        for obs in tr:
            by_time[obs["time_sec"]][tid] = obs

    pair_stats = defaultdict(lambda: {"events":0, "follow":0, "stable":0, "dists":[]})
    times = sorted(by_time.keys())
    prev_by_tid = {}
    for t in times:
        obs = by_time[t]
        tids = list(obs.keys())
        for i in range(len(tids)):
            for j in range(i+1, len(tids)):
                a, b = tids[i], tids[j]
                ca, cb = tuple(obs[a]["center"]), tuple(obs[b]["center"])
                d = dist(ca, cb)
                if not (FOLLOW_DISTANCE_MIN <= d <= FOLLOW_DISTANCE_MAX):
                    continue
                total_pair_events += 1
                key = tuple(sorted((a,b)))
                pair_stats[key]["events"] += 1
                pair_stats[key]["dists"].append(d)
                if a in prev_by_tid and b in prev_by_tid:
                    va = vec(tuple(prev_by_tid[a]["center"]), ca)
                    vb = vec(tuple(prev_by_tid[b]["center"]), cb)
                    cos = cosine(va, vb)
                    if cos >= FOLLOW_DIRECTION_COS_THRESHOLD and norm(va) > 5 and norm(vb) > 5:
                        follow_events += 1
                        pair_stats[key]["follow"] += 1
        prev_by_tid = obs

    for key, st in pair_stats.items():
        if len(st["dists"]) >= 3:
            if np.std(st["dists"]) <= FOLLOW_DISTANCE_STABILITY:
                stable_events += st["events"]
                st["stable"] = st["events"]
        score = 0
        score += min(0.45, st["follow"] / max(1, st["events"]) * 0.45)
        score += min(0.25, st["events"] / max(1, len(times)) * 0.25)
        if st["stable"] > 0:
            score += 0.20
        if score > best_pair_score:
            best_pair_score = score
            best_pair = {"pair": list(key), **st, "score": round(score,4)}

    score = 0.0
    score += min(0.50, best_pair_score)
    score += min(0.25, follow_events / max(1, total_pair_events) * 0.25)
    score += min(0.15, stable_events / max(1, total_pair_events) * 0.15)
    score += min(0.10, long_track_bonus * 0.04)
    score = round(float(min(1.0, score)), 4)

    return {
        "stalking_score": score,
        "is_stalking": score >= STALKING_THRESHOLD,
        "follow_events": follow_events,
        "total_pair_events": total_pair_events,
        "stable_events": stable_events,
        "long_track_bonus": long_track_bonus,
        "best_pair": best_pair,
    }


def window_iter(frames, window_seconds=5.0, stride_seconds=1.5):
    if not frames:
        return []
    start = frames[0].get("time_sec", 0)
    end = frames[-1].get("time_sec", start)
    wins = []
    t = start
    while t <= end:
        w_frames = [fr for fr in frames if t <= fr.get("time_sec", 0) < t + window_seconds]
        if w_frames:
            wins.append((t, t + window_seconds, w_frames))
        t += stride_seconds
    return wins


def compute_fight_window_score(w_frames):
    close_pairs = 0
    multi_person = 0
    overlap_events = 0
    reversal_events = 0
    pose_vals = []
    flow_vals = []
    aggressive_motion = 0

    prev_centers_sorted = None
    for fr in w_frames:
        persons = fr.get("persons", [])
        if len(persons) >= 2:
            multi_person += 1
            centers = [tuple(p["center"]) for p in persons]
            min_d = 10**9
            for i in range(len(persons)):
                for j in range(i+1, len(persons)):
                    d = dist(centers[i], centers[j])
                    min_d = min(min_d, d)
                    if d < 180:
                        close_pairs += 1
                    if bbox_iou(persons[i]["bbox"], persons[j]["bbox"]) > 0.02:
                        overlap_events += 1
            centers_sorted = sorted(centers, key=lambda x: x[0])[:2]
            if prev_centers_sorted is not None and len(centers_sorted) >= 2:
                # 좌우 순서 변화는 몸이 뒤섞이는 싸움/충돌 후보
                prev_gap = prev_centers_sorted[1][0] - prev_centers_sorted[0][0]
                curr_gap = centers_sorted[1][0] - centers_sorted[0][0]
                if prev_gap * curr_gap < 0:
                    reversal_events += 1
            prev_centers_sorted = centers_sorted
        pose_vals.append(fr.get("pose_agitation", 0.0))
        fl = fr.get("flow", {})
        flow_vals.append(fl.get("p90_mag", 0.0))
        if fl.get("strong_ratio", 0.0) > 0.08:
            aggressive_motion += 1

    n = max(1, len(w_frames))
    multi_ratio = multi_person / n
    close_ratio = min(1.0, close_pairs / max(1, n))
    overlap_ratio = min(1.0, overlap_events / max(1, n))
    pose_score = min(1.0, float(np.mean(pose_vals)) / 2.8) if pose_vals else 0.0
    flow_score = min(1.0, float(np.mean(flow_vals)) / 8.0) if flow_vals else 0.0
    aggressive_ratio = aggressive_motion / n
    reversal_ratio = min(1.0, reversal_events / max(1, n))

    # 싸움은 단순 다인/근접보다, 근접 + 움직임 + pose/flow의 동시성에 가중
    score = 0.0
    score += 0.10 * multi_ratio
    score += 0.22 * close_ratio
    score += 0.14 * overlap_ratio
    score += 0.18 * pose_score
    score += 0.22 * flow_score
    score += 0.10 * aggressive_ratio
    score += 0.04 * reversal_ratio

    # 게이트: 가까운 사람이 없으면 flow만으로 싸움으로 보지 않음
    if close_ratio < 0.08:
        score *= 0.55
    # pose 모델이 없을 때 너무 불리하지 않도록 보정
    if pose_model is None:
        score += 0.05 * close_ratio + 0.05 * aggressive_ratio

    return {
        "score": round(float(min(1.0, score)), 4),
        "multi_ratio": round(multi_ratio, 4),
        "close_ratio": round(close_ratio, 4),
        "overlap_ratio": round(overlap_ratio, 4),
        "pose_score": round(pose_score, 4),
        "flow_score": round(flow_score, 4),
        "aggressive_ratio": round(aggressive_ratio, 4),
        "reversal_ratio": round(reversal_ratio, 4),
    }


def compute_fight_score(frames):
    windows = []
    best = None
    for s, e, w_frames in window_iter(frames, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS):
        st = compute_fight_window_score(w_frames)
        item = {"start_time": round(s,2), "end_time": round(e,2), **st}
        windows.append(item)
        if best is None or item["score"] > best["score"]:
            best = item
    peak = best["score"] if best else 0.0
    return {
        "fight_score": peak,
        "is_fight": peak >= FIGHT_THRESHOLD,
        "best_fight_window": best,
        "windows": windows,
    }



def compute_riding_score(frames):
    """
    aio2 강화 버전.
    - 전체 평균만 보지 않고, 프레임별 peak evidence를 강하게 반영
    - YOLO-World 검출을 기본 YOLO 보조 증거로 적극 반영
    - rider_like만 있어도 이륜차 후보로 인정
    - SegFormer가 잡은 인도 비율은 mean보다 peak를 더 크게 반영
    """
    tw_frames = 0
    rider_frames = 0
    sidewalk_ratios = []
    road_ratios = []
    seg_used_count = 0

    world_tw_count = 0
    base_tw_count = 0
    total_tw_count = 0
    high_conf_tw_count = 0

    rider_distances = []
    per_frame_detection_scores = []
    per_frame_sidewalk_scores = []

    for fr in frames:
        tw = fr.get("two_wheelers", [])
        rider_pairs = fr.get("rider_pairs", [])

        if tw:
            tw_frames += 1
        if rider_pairs:
            rider_frames += 1
        if fr.get("segformer_used"):
            seg_used_count += 1

        frame_tw_score = 0.0

        for obj in tw:
            total_tw_count += 1
            cls = str(obj.get("class", "")).lower()
            conf = float(obj.get("confidence", 0) or 0)

            if cls in TWO_WHEELER_CLASSES:
                base_tw_count += 1
                frame_tw_score += min(0.45, conf * 1.2)
            else:
                world_tw_count += 1
                frame_tw_score += min(0.40, conf * 2.0)

            if conf >= 0.20:
                high_conf_tw_count += 1

        # rider pair 자체도 강한 보조 증거로 사용
        if rider_pairs:
            frame_tw_score += 0.25
            for pair in rider_pairs:
                d = pair.get("distance")
                if d is not None:
                    rider_distances.append(float(d))

        per_frame_detection_scores.append(min(1.0, frame_tw_score))

        frame_sidewalk_scores = []
        for ev in fr.get("sidewalk_evidence", []):
            sr = ev.get("sidewalk_ratio")
            rr = ev.get("road_ratio")

            if sr is not None:
                sidewalk_ratios.append(float(sr))
                frame_sidewalk_scores.append(float(sr))
            if rr is not None:
                road_ratios.append(float(rr))

        per_frame_sidewalk_scores.append(max(frame_sidewalk_scores) if frame_sidewalk_scores else 0.0)

    n = max(1, len(frames))
    two_wheeler_ratio = tw_frames / n
    rider_ratio = rider_frames / n

    sidewalk_mean = float(np.mean(sidewalk_ratios)) if sidewalk_ratios else 0.0
    road_mean = float(np.mean(road_ratios)) if road_ratios else 0.0
    sidewalk_peak = float(np.max(sidewalk_ratios)) if sidewalk_ratios else 0.0

    detection_mean = float(np.mean(per_frame_detection_scores)) if per_frame_detection_scores else 0.0
    detection_peak = float(np.max(per_frame_detection_scores)) if per_frame_detection_scores else 0.0

    sidewalk_frame_mean = float(np.mean(per_frame_sidewalk_scores)) if per_frame_sidewalk_scores else 0.0
    sidewalk_frame_peak = float(np.max(per_frame_sidewalk_scores)) if per_frame_sidewalk_scores else 0.0

    detection_score = min(
        1.0,
        two_wheeler_ratio * 0.35
        + rider_ratio * 0.30
        + detection_mean * 0.25
        + detection_peak * 0.45
        + min(0.20, world_tw_count * 0.015)
        + min(0.15, base_tw_count * 0.010)
    )

    sidewalk_score = min(
        1.0,
        sidewalk_mean * 0.35
        + sidewalk_peak * 0.45
        + sidewalk_frame_peak * 0.20
    )

    if detection_score < RIDE_DETECTION_THRESHOLD:
        riding_label = "no_two_wheeler"
    elif sidewalk_score >= RIDE_SIDEWALK_THRESHOLD and sidewalk_peak >= 0.20:
        riding_label = "two_wheeler_sidewalk_riding"
    elif sidewalk_score >= RIDE_SIDEWALK_CANDIDATE_THRESHOLD or sidewalk_peak >= 0.16:
        riding_label = "two_wheeler_sidewalk_candidate"
    else:
        riding_label = "two_wheeler_rider_like"

    return {
        "riding_label": riding_label,
        "two_wheeler_detected": detection_score >= RIDE_DETECTION_THRESHOLD,
        "two_wheeler_ratio": round(two_wheeler_ratio, 4),
        "rider_ratio": round(rider_ratio, 4),
        "detection_score": round(detection_score, 4),
        "detection_mean": round(detection_mean, 4),
        "detection_peak": round(detection_peak, 4),
        "sidewalk_score": round(sidewalk_score, 4),
        "sidewalk_mean": round(sidewalk_mean, 4),
        "sidewalk_peak": round(sidewalk_peak, 4),
        "sidewalk_frame_mean": round(sidewalk_frame_mean, 4),
        "sidewalk_frame_peak": round(sidewalk_frame_peak, 4),
        "road_mean": round(road_mean, 4),
        "segformer_frame_count": seg_used_count,
        "base_two_wheeler_object_count": base_tw_count,
        "world_two_wheeler_object_count": world_tw_count,
        "total_two_wheeler_object_count": total_tw_count,
        "high_conf_two_wheeler_count": high_conf_tw_count,
        "avg_rider_distance": round(float(np.mean(rider_distances)), 2) if rider_distances else None,
        "min_rider_distance": round(float(np.min(rider_distances)), 2) if rider_distances else None,
    }

def decide_action_label(fight, stalking):
    # 동시에 높으면 둘 다 태그로 남기되, action_label은 더 높은 쪽
    fs = fight.get("fight_score", 0)
    ss = stalking.get("stalking_score", 0)
    if fight.get("is_fight") and stalking.get("is_stalking"):
        return "fight" if fs >= ss else "stalking"
    if fight.get("is_fight"):
        return "fight"
    if stalking.get("is_stalking"):
        return "stalking"
    return "other"


def make_final_tags(action_label, fight, stalking, riding):
    tags = []
    if action_label != "other":
        tags.append(action_label)
    else:
        tags.append("other")
    if fight.get("is_fight"):
        tags.append("fight")
    if stalking.get("is_stalking"):
        tags.append("stalking")
    rl = riding.get("riding_label", "no_two_wheeler")
    if rl != "no_two_wheeler":
        tags.append("two_wheeler_detected")
        tags.append(rl)
    # 중복 제거, 순서 유지
    out = []
    for t in tags:
        if t not in out:
            out.append(t)
    return out


In [7]:

# ============================================================
# Cell 7. AIO 최종 판정 실행
# ============================================================
feature_json_paths = sorted(AIO_FEATURE_DIR.rglob("*.json"))
print("feature json 수:", len(feature_json_paths))

results = []
failed = []

for jp in tqdm(feature_json_paths, desc="AIO final classify"):
    try:
        with open(jp, "r", encoding="utf-8") as f:
            data = json.load(f)
        frames = data.get("frames", [])
        tracks = assign_tracks(frames)
        stalking = compute_stalking_score(frames, tracks)
        fight = compute_fight_score(frames)
        riding = compute_riding_score(frames)
        action_label = decide_action_label(fight, stalking)
        final_tags = make_final_tags(action_label, fight, stalking, riding)

        # 병렬 판정이므로 최종 라벨은 덮어쓰기보다 조합 문자열
        final_label = " + ".join(final_tags) if final_tags else "other"
        if not final_label or final_label == "unknown":
            final_label = "other"

        track_summary = {
            str(tid): {**track_motion(tr), "first_time": tr[0]["time_sec"], "last_time": tr[-1]["time_sec"]}
            for tid, tr in tracks.items()
        }

        item = {
            "video_name": data.get("video_name"),
            "video_path": data.get("video_path"),
            "relative_video_path": data.get("relative_video_path"),
            "final_label": final_label,
            "final_tags": final_tags,
            "action_label": action_label,
            "fight_label": "fight" if fight.get("is_fight") else "not_fight",
            "stalking_label": "stalking" if stalking.get("is_stalking") else "not_stalking",
            "riding_label": riding.get("riding_label"),
            "fight": {k:v for k,v in fight.items() if k != "windows"},
            "stalking": stalking,
            "riding": riding,
            "track_summary": track_summary,
            "model_status": MODEL_STATUS,
            "feature_json": str(jp),
        }
        results.append(item)
    except Exception as e:
        failed.append({"json": str(jp), "error": repr(e)})

summary = {
    "task": "aio2_multi_model_parallel_classification_enhanced_riding",
    "video_root": str(VIDEO_ROOT),
    "output_dir": str(AIO_DIR),
    "feature_dir": str(AIO_FEATURE_DIR),
    "model_status": MODEL_STATUS,
    "settings": {
        "frame_interval": FRAME_INTERVAL,
        "fight_threshold": FIGHT_THRESHOLD,
        "stalking_threshold": STALKING_THRESHOLD,
        "ride_detection_threshold": RIDE_DETECTION_THRESHOLD,
        "ride_sidewalk_threshold": RIDE_SIDEWALK_THRESHOLD,
        "window_seconds": WINDOW_SECONDS,
        "window_stride_seconds": WINDOW_STRIDE_SECONDS,
    },
    "total_json": len(feature_json_paths),
    "success_count": len(results),
    "failed_count": len(failed),
    "final_label_count": dict(Counter(r["final_label"] for r in results)),
    "action_label_count": dict(Counter(r["action_label"] for r in results)),
    "riding_label_count": dict(Counter(r["riding_label"] for r in results)),
    "results": results,
    "failed": failed,
}

summary_path = AIO_FINAL_DIR / "aio2_final_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("="*80)
print("AIO 최종 판정 완료")
print("성공:", len(results), "실패:", len(failed))
print("summary 저장:", summary_path)
print("final_label_count:", summary["final_label_count"])
print("action_label_count:", summary["action_label_count"])
print("riding_label_count:", summary["riding_label_count"])
if failed:
    for x in failed[:10]:
        print(x)


feature json 수: 22


AIO final classify: 100%|██████████████████████████████████████████████████████████████| 22/22 [00:02<00:00,  8.20it/s]

AIO 최종 판정 완료
성공: 22 실패: 0
summary 저장: C:\Users\우성한\outputs\prediction\aio2\final_aio\aio2_final_summary.json
final_label_count: {'stalking + two_wheeler_detected + two_wheeler_rider_like': 1, 'stalking + two_wheeler_detected + two_wheeler_sidewalk_candidate': 1, 'stalking + fight + two_wheeler_detected + two_wheeler_rider_like': 3, 'stalking + two_wheeler_detected + two_wheeler_sidewalk_riding': 2, 'stalking + fight + two_wheeler_detected + two_wheeler_sidewalk_riding': 4, 'other + two_wheeler_detected + two_wheeler_rider_like': 5, 'other + two_wheeler_detected + two_wheeler_sidewalk_riding': 4, 'other + two_wheeler_detected + two_wheeler_sidewalk_candidate': 2}
action_label_count: {'stalking': 11, 'other': 11}
riding_label_count: {'two_wheeler_rider_like': 9, 'two_wheeler_sidewalk_candidate': 3, 'two_wheeler_sidewalk_riding': 10}


In [8]:

# ============================================================
# Cell 8. 결과를 알아보기 쉽게 출력 + txt 저장
# ============================================================
summary_path = AIO_FINAL_DIR / "aio2_final_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)
results = summary.get("results", [])

print("="*100)
print("AIO2 결과 요약")
print("="*100)
print("전체 결과 수:", len(results))
print("모델 상태:", summary.get("model_status"))

print("\n[최종 라벨별 개수]")
for k, v in Counter(r.get("final_label", "unknown") for r in results).most_common():
    print(f"{k:60s}: {v}")

print("\n[행동 라벨별 개수]")
for k, v in Counter(r.get("action_label", "unknown") for r in results).most_common():
    print(f"{k:30s}: {v}")

print("\n[주행 라벨별 개수]")
for k, v in Counter(r.get("riding_label", "unknown") for r in results).most_common():
    print(f"{k:40s}: {v}")

# 싸움 점수 top
fight_rank = sorted(results, key=lambda r: r.get("fight", {}).get("fight_score", 0), reverse=True)
print("\n" + "="*100)
print("싸움 점수 TOP 20")
print("="*100)
for i, r in enumerate(fight_rank[:20], start=1):
    f = r.get("fight", {})
    w = f.get("best_fight_window") or {}
    print(f"{i:02d}. {r.get('video_name')} | {r.get('relative_video_path')}")
    print("    final:", r.get("final_label"), "action:", r.get("action_label"), "ride:", r.get("riding_label"))
    print("    fight_score:", f.get("fight_score"), "window:", w.get("start_time"), "~", w.get("end_time"))

# 스토킹 점수 top
stalk_rank = sorted(results, key=lambda r: r.get("stalking", {}).get("stalking_score", 0), reverse=True)
print("\n" + "="*100)
print("스토킹 점수 TOP 20")
print("="*100)
for i, r in enumerate(stalk_rank[:20], start=1):
    s = r.get("stalking", {})
    print(f"{i:02d}. {r.get('video_name')} | {r.get('relative_video_path')}")
    print("    final:", r.get("final_label"), "action:", r.get("action_label"), "ride:", r.get("riding_label"))
    print("    stalking_score:", s.get("stalking_score"), "follow_events:", s.get("follow_events"), "best_pair:", s.get("best_pair"))

# 인도주행 후보 top
ride_rank = sorted(results, key=lambda r: r.get("riding", {}).get("sidewalk_score", 0), reverse=True)
print("\n" + "="*100)
print("인도주행/침범 점수 TOP 20")
print("="*100)
for i, r in enumerate(ride_rank[:20], start=1):
    rd = r.get("riding", {})
    print(f"{i:02d}. {r.get('video_name')} | {r.get('relative_video_path')}")
    print("    final:", r.get("final_label"), "ride:", r.get("riding_label"))
    print("    sidewalk_score:", rd.get("sidewalk_score"), "sidewalk_mean:", rd.get("sidewalk_mean"), "road_mean:", rd.get("road_mean"), "tw_ratio:", rd.get("two_wheeler_ratio"))

print("\n" + "="*100)
print("영상별 최종 판정")
print("="*100)
for i, r in enumerate(results, start=1):
    print("-"*100)
    print(f"[{i}] {r.get('video_name')}")
    print("상대 경로:", r.get("relative_video_path"))
    print("최종 판정:", r.get("final_label"))
    print("태그:", ", ".join(r.get("final_tags", [])))
    print("행동:", r.get("action_label"), "| 싸움:", r.get("fight", {}).get("fight_score"), "| 스토킹:", r.get("stalking", {}).get("stalking_score"))
    print("주행:", r.get("riding_label"), "| sidewalk_score:", r.get("riding", {}).get("sidewalk_score"))

# txt 저장
lines = []
lines.append("="*100)
lines.append("AIO2 결과 요약")
lines.append("="*100)
lines.append(f"전체 결과 수: {len(results)}")
lines.append(f"모델 상태: {summary.get('model_status')}")
lines.append("")
lines.append("[최종 라벨별 개수]")
for k, v in Counter(r.get("final_label", "unknown") for r in results).most_common():
    lines.append(f"{k:60s}: {v}")
lines.append("")
lines.append("[영상별 최종 판정]")
for i, r in enumerate(results, start=1):
    lines.append("-"*100)
    lines.append(f"[{i}] {r.get('video_name')}")
    lines.append(f"상대 경로: {r.get('relative_video_path')}")
    lines.append(f"최종 판정: {r.get('final_label')}")
    lines.append(f"태그: {', '.join(r.get('final_tags', []))}")
    lines.append(f"행동: {r.get('action_label')} | 싸움점수: {r.get('fight', {}).get('fight_score')} | 스토킹점수: {r.get('stalking', {}).get('stalking_score')}")
    lines.append(f"주행: {r.get('riding_label')} | sidewalk_score: {r.get('riding', {}).get('sidewalk_score')}")

txt_path = AIO_FINAL_DIR / "aio2_readable_result.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\nTXT 저장:", txt_path)


AIO2 결과 요약
전체 결과 수: 22
모델 상태: {'base_yolo': 'yolov8n.pt', 'pose_model': 'yolo11n-pose.pt', 'world_model': 'yolov8s-world.pt', 'segformer': 'nvidia/segformer-b5-finetuned-cityscapes-1024-1024', 'helmet_model': None}

[최종 라벨별 개수]
other + two_wheeler_detected + two_wheeler_rider_like       : 5
stalking + fight + two_wheeler_detected + two_wheeler_sidewalk_riding: 4
other + two_wheeler_detected + two_wheeler_sidewalk_riding  : 4
stalking + fight + two_wheeler_detected + two_wheeler_rider_like: 3
stalking + two_wheeler_detected + two_wheeler_sidewalk_riding: 2
other + two_wheeler_detected + two_wheeler_sidewalk_candidate: 2
stalking + two_wheeler_detected + two_wheeler_rider_like    : 1
stalking + two_wheeler_detected + two_wheeler_sidewalk_candidate: 1

[행동 라벨별 개수]
stalking                      : 11
other                         : 11

[주행 라벨별 개수]
two_wheeler_sidewalk_riding             : 10
two_wheeler_rider_like                  : 9
two_wheeler_sidewalk_candidate          : 3

싸움 점수 TOP 2

In [9]:

# ============================================================
# Cell 9. 자연어 비슷한 검색 함수
# ============================================================
def search_aio2_results(query, top_k=50):
    q = str(query).lower()
    summary_path = AIO_FINAL_DIR / "aio2_final_summary.json"
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    results = summary.get("results", [])

    def match(r):
        hay = " ".join([
            str(r.get("video_name", "")),
            str(r.get("relative_video_path", "")),
            str(r.get("final_label", "")),
            str(r.get("action_label", "")),
            str(r.get("riding_label", "")),
            " ".join(r.get("final_tags", [])),
        ]).lower()
        # 한글 키워드 매핑
        if "싸움" in q or "폭행" in q:
            return r.get("fight", {}).get("is_fight") or r.get("action_label") == "fight" or "fight" in hay
        if "스토킹" in q or "배회" in q or "추적" in q:
            return r.get("stalking", {}).get("is_stalking") or r.get("action_label") == "stalking" or "stalking" in hay
        if "인도" in q or "침범" in q:
            return "sidewalk" in str(r.get("riding_label", "")).lower()
        if "이륜" in q or "오토바이" in q or "자전거" in q or "주행" in q:
            return r.get("riding_label") != "no_two_wheeler"
        return q in hay

    matched = [r for r in results if match(r)]
    print(f"검색어: {query}")
    print(f"검색 결과: {len(matched)}개")
    for i, r in enumerate(matched[:top_k], start=1):
        print("-"*80)
        print(f"[{i}] {r.get('video_name')}")
        print("상대 경로:", r.get("relative_video_path"))
        print("최종 판정:", r.get("final_label"))
        print("행동:", r.get("action_label"), "싸움점수:", r.get("fight", {}).get("fight_score"), "스토킹점수:", r.get("stalking", {}).get("stalking_score"))
        print("주행:", r.get("riding_label"), "sidewalk_score:", r.get("riding", {}).get("sidewalk_score"))
    return matched

# 예시:
# search_aio2_results("싸움")
# search_aio_results("스토킹")
# search_aio_results("인도 주행")
# search_aio_results("이륜차")


In [10]:
# ============================================================
# Cell 10. AIO2 이륜차/킥보드 검출 디버그 출력
# ============================================================
import json
from pathlib import Path
from collections import Counter

summary_path = AIO_FINAL_DIR / "aio2_final_summary.json"

if not summary_path.exists():
    raise FileNotFoundError(f"summary 파일이 없습니다: {summary_path}")

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

results = summary.get("results", [])

print("=" * 100)
print("AIO2 주행/이륜차 라벨 개수")
print("=" * 100)
for k, v in Counter(r.get("riding_label", "unknown") for r in results).most_common():
    print(f"{k:45s}: {v}")

print("\n" + "=" * 100)
print("이륜차 검출 점수 높은 순 TOP 30")
print("=" * 100)

ranked = sorted(
    results,
    key=lambda r: r.get("riding", {}).get("detection_score", 0),
    reverse=True
)

for i, r in enumerate(ranked[:30], start=1):
    rd = r.get("riding", {})
    print()
    print(f"[{i}] {r.get('video_name')}")
    print("상대 경로:", r.get("relative_video_path"))
    print("최종 판정:", r.get("final_label"))
    print("주행 라벨:", r.get("riding_label"))
    print("detection_score:", rd.get("detection_score"))
    print("detection_peak:", rd.get("detection_peak"))
    print("detection_mean:", rd.get("detection_mean"))
    print("two_wheeler_ratio:", rd.get("two_wheeler_ratio"))
    print("rider_ratio:", rd.get("rider_ratio"))
    print("base_tw_count:", rd.get("base_two_wheeler_object_count"))
    print("world_tw_count:", rd.get("world_two_wheeler_object_count"))
    print("total_tw_count:", rd.get("total_two_wheeler_object_count"))
    print("sidewalk_score:", rd.get("sidewalk_score"))
    print("sidewalk_peak:", rd.get("sidewalk_peak"))
    print("segformer_frame_count:", rd.get("segformer_frame_count"))

print("\n" + "=" * 100)
print("주의")
print("=" * 100)
print("기존 aio2_features가 이미 있으면 새 설정이 반영되지 않을 수 있습니다.")
print("설정을 바꾼 뒤 재처리하려면 aio2/aio_features 폴더를 삭제한 후 Cell 5부터 다시 실행하세요.")

AIO2 주행/이륜차 라벨 개수
two_wheeler_sidewalk_riding                  : 10
two_wheeler_rider_like                       : 9
two_wheeler_sidewalk_candidate               : 3

이륜차 검출 점수 높은 순 TOP 30

[1] lo_e1281_c1.mp4
상대 경로: 스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
최종 판정: stalking + two_wheeler_detected + two_wheeler_rider_like
주행 라벨: two_wheeler_rider_like
detection_score: 1.0
detection_peak: 1.0
detection_mean: 0.2811
two_wheeler_ratio: 0.7198
rider_ratio: 0.6264
base_tw_count: 275
world_tw_count: 0
total_tw_count: 275
sidewalk_score: 0.0
sidewalk_peak: 0.0
segformer_frame_count: 182

[2] lo_e1281_c2.mp4
상대 경로: 스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
최종 판정: stalking + two_wheeler_detected + two_wheeler_sidewalk_candidate
주행 라벨: two_wheeler_sidewalk_candidate
detection_score: 1.0
detection_peak: 1.0
detection_mean: 0.6071
two_wheeler_ratio: 0.9516
rider_ratio: 0.828
base_tw_count: 687
world_tw_count: 0
total_tw_count: 687
sidewalk_score: 0.1668
sidewalk_peak: 0.2549
segformer_frame_count